# PCA Project: Abalone Dataset
## Goal: Understand and implement PCA from scratch

---
## Phase 1: Setup & Data Loading

# 1. Import Libraries

**Why?** We need specialized tools for mathematical operations and data processing.

**How?** Import standard data science libraries: `numpy` (matrices/linear algebra), `pandas` (dataframes), `matplotlib/seaborn` (visualization), and `scipy` (statistics).

**In Code:** Load necessary modules into the working environment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
# Ignore warnings containing "batch_size"
warnings.filterwarnings("ignore", message=".*batch_size.*")

# 2. Load Data

**Why?** The dataset is available as a CSV file and needs to be loaded into memory for analysis.

**How?** Use `pd.read_csv` to parse the comma-separated values into a Pandas DataFrame.

**In Code:** Read `abalone.csv` from the Data folder.

In [ ]:
# Load data
abalone = pd.read_csv(r"Data/abalone.csv")

# 3. First Overview

**Why?** We need to understand the structure, column names, and data types to choose the right preprocessing steps.

**How?** `head()` shows the first 5 rows, `info()` shows data types and memory usage.

**In Code:** Display raw data and metadata.

In [ ]:
# First look at the data
display(abalone.head())
print(f"Shape of dataset: {abalone.shape}")
print("Data Types:")
print(abalone.info())

---
## Phase 2: Exploratory Data Analysis (EDA)

# 4. Descriptive Statistics

**Why?** Get a sense of the distributions, means, and spread of the data.

**How?** `describe()` calculates metrics like Mean, Std, Min, Max, and Quartiles for numeric columns.

**In Code:** Statistical summary of the dataset.

In [ ]:
# Descriptive statistics
print(abalone.describe())

# 5. Check for Missing Values

**Why?** PCA is a purely mathematical method and cannot handle missing values (NaN). These would need to be replaced or removed.

**How?** Check for null values with `isnull().sum()`.

**In Code:** Ensure data quality.

In [ ]:
# Create histograms
abalone.hist(bins=50, figsize=(20,15))
plt.show()

In [ ]:
# Check for missing values
print (abalone.isnull().sum())

In [ ]:
# Select numeric features
numeric_cols = abalone.select_dtypes(include=['number']).columns
# Erstelle ein Grid aus einzelnen Plots
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()
# Jeden Plot einzeln erstellen -> Garantiert eigene Achsen!
for i, col in enumerate(numeric_cols):
    sns.histplot(data=abalone, x=col, kde=True, ax=axes[i], color='steelblue')
    axes[i].set_title(col, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('')
    
# Leere Subplots ausblenden
for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Q-Q Plots

# Create a figure with 3x3 grid of subplots (9 total)
# figsize sets the overall size in inches (width, height)
fig, axes = plt.subplots(3, 3, figsize=(12, 10))

# Flatten the 2D array of axes into 1D for easier indexing
# Before: [[ax0,ax1,ax2], [ax3,ax4,ax5], [ax6,ax7,ax8]]
# After:  [ax0, ax1, ax2, ax3, ax4, ax5, ax6, ax7, ax8]
axes = axes.flatten()

# Loop through each numeric column with its index
# enumerate gives us: (0,'Length'), (1,'Diameter'), (2,'Height'), ...
for i, col in enumerate(numeric_cols):
    # Create Q-Q plot for this column on the i-th subplot
    stats.probplot(abalone[col], plot=axes[i])
    # Set the title of this subplot to the column name
    axes[i].set_title(col)

# Adjust spacing between subplots to prevent overlap
plt.tight_layout()
# Display the figure
plt.show()
#Length, Diameter and Height show a normal distribution
#whole weight,shucked weight and viscera weight dont seem to be normal distributed
    

In [ ]:

# D'Agostino-Pearson Test - better for large samples
for col in numeric_cols:
    stat, p = stats.normaltest(abalone[col])
    result = "Normal" if p > 0.05 else "Not normal"
    print(f"{col}: p={p:.2e} = {result}")

print (stat)
print (p)

In [ ]:
#height has some weird outliers that we should remove
#before:

abalone.drop(abalone[abalone["Height"] >0.4].index, inplace=True)
print (f"new shape: {abalone.shape}")

In [ ]:
# encode sex - Kategorische Variable (M/F/I) enkodieren
abalone["Sex"] = abalone["Sex"].map({"M": 0, "F": 1, "I": 2})
print(abalone.head())

In [ ]:
# correlation analysis - Korrelationsmatrix & Heatmap
sns.heatmap(abalone.drop(columns='Sex').corr(), annot=True)
#exclude sex column 
plt.show()

# Correlation matrix: high values between features = redundancy (good for PCA)
# Shell weight has best correlation with Rings (0.63) = best predictor for age

---
## Phase 3: Prepare Data for PCA

In [ ]:
# select numeric features 
abalone_num = abalone.select_dtypes(include=["int64", "float64"])
print(abalone_num.head())

In [ ]:
# standardize data 
abalone_standardized = (abalone - abalone.mean()) / abalone.std()
print(abalone_standardized.head())

---
## Phase 4: Understand PCA Mathematics
Here we implement PCA step by step with NumPy

### 1. Data Standardization & Covariance Matrix

**Why standardize?**
PCA is sensitive to the scaling of the data. Features with large value ranges (e.g. weight) would dominate the analysis. Therefore, we have already z-standardized the data (mean = 0, std = 1): `abalone_standardized`.

**The Covariance Matrix ($\Sigma$)**
Now we calculate the covariance matrix. It describes how the features vary together (correlate).
Mathematically, this is calculated from the standardized data matrix $X$:

$$ \Sigma = \frac{1}{n-1} X^T X $$

The result is a $d \times d$ matrix (d = number of features) that is symmetric.

In [ ]:
# covariance matrix - Calculate covariance matrix (manually with NumPy)
# IMPORTANT: We use the standardized data (abalone_standardized), not the raw numbers!
abalone_cov = np.cov(abalone_standardized.T)
print(abalone_cov)

### 2. Eigendecomposition

We are now looking for the 'principal axes' of the data. These are the directions in which the data has the greatest variance (information).
Mathematically we solve the **eigenvalue problem** for the covariance matrix $\Sigma$:

$$ \Sigma v = \lambda v $$

*   **Eigenvector ($v$)**: Indicates the direction of the new axis (principal component).
*   **Eigenvalue ($\lambda$)**: Indicates the magnitude of the variance along this axis.

![PCA Covariance and Eigenvectors](pca_covariance.png)

High Eigenvalue = Much Information.
Low Eigenvalue = Little Information (Noise).

In [ ]:
# eigenvalues eigenvectors - Calculate eigenvalues & eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(abalone_cov)
print("Eigenvalues:\n", eigenvalues)
print("\nEigenvectors:\n", eigenvectors)

### 3. Sorting Principal Components (Feature Ranking)

**Why sort?**
Not all directions are equally important. The eigenvalues tell us how much variance (information) is in each direction.
We want to reduce dimensions by keeping only the *most important* directions.
Therefore, we sort the eigenvectors in descending order by their eigenvalues.

*   Highest Eigenvalue $\rightarrow$ PC1 (Most Important Component)
*   Second Highest Eigenvalue $\rightarrow$ PC2
*   ...

In [ ]:
# sort components - Sort by variance
# We use argsort to get the indices from sorting
idx = eigenvalues.argsort()[::-1]

eigenvalues_sorted = eigenvalues[idx]
eigenvectors_sorted = eigenvectors[:, idx]

print("Sorted Eigenvalues:", eigenvalues_sorted)
print("Sorted Eigenvectors:", eigenvectors_sorted)

### 4. Explained Variance Ratio

**How important is each component?**
We want to know what percentage of the original information (total variance) is contained in each principal component.

$$ \text{Explained Variance Ratio} = \frac{\lambda_i}{\sum_{j=1}^{d} \lambda_j} $$

This helps us decide how many dimensions we want to keep (e.g. "We keep 95% of information").

In [ ]:
# explained variance - Explained variance per component
explained_variance = (eigenvalues_sorted / np.sum(eigenvalues_sorted)*100)
print (explained_variance)
# Component 1 explains 78% of the variance, component 2 explains 8.6%, which together explain over 87%, meaning it's enough for a 2D plot
 # we take the first two eigenvectors as our projection matrix
projection_matrix = eigenvectors_sorted[:, :2]
print (projection_matrix)

---
## Phase 5: Implement PCA yourself

### 3. Data Projection

The last step is the transformation of the original data into the new coordinate system.
To do this, we multiply our data matrix $X$ by the matrix of the selected eigenvectors $W$ (feature matrix).

$$ Z = X \cdot W $$

![PCA Projection Concept](pca_projection.png)

The result $Z$ are the so-called **Principal Components** (Scores).

In [ ]:
# project data - Project data onto new axes
abalone_pca = abalone_standardized.dot(projection_matrix)
abalone_pca.columns = ['PC1', 'PC2']
print (abalone_pca.head())

print("Old Dimensions:", abalone_standardized.shape)
print("Neue Dimensionen:", abalone_pca.shape) #  (4175, 2) insteas od 4175, 9


In [ ]:
# scree plot - Visualize variance proportion
plt.figure(figsize=(10, 6))
# We have 9 components (number of features)
r = range(1, len(explained_variance) + 1)
plt.plot(r, explained_variance, marker='o', linestyle='--')
plt.title('Scree Plot')
plt.xlabel('Maincomponent')
plt.ylabel('Explained Variance (%)')
plt.grid(True)
plt.show()

In [ ]:
# Mappe die Zahlen zurück zu Buchstaben für die Legende
sex_labels = abalone['Sex'].map({0: 'M', 1: 'F', 2: 'I'})

plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='PC1', 
    y='PC2', 
    data=abalone_pca, 
    hue=abalone['Rings'], 
    style=sex_labels,        # Hier nutzen wir die Buchstaben
    palette='viridis', 
    alpha=0.6
)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA Projection colorcoded by Rings")
plt.legend(title='Rings')
plt.show()

---
## Phase 6: Vergleich mit sklearn

# 15. Vergleich mit Scikit-Learn

**Wieso?** Validierung unserer manuellen Berechnung und Nutzung effizienter Standard-Tools für die Praxis.

**Wie?** Nutzung der `PCA` Klasse aus `sklearn.decomposition`. Das macht intern genau das Gleiche (oft via SVD).

**Im Code:** Anwendung der Standard-Implementierung zum Vergleich.

In [ ]:
# sklearn pca - sklearn.decomposition.PCA anwenden
from sklearn.decomposition import PCA
# "Maschine" bauen, die nur die 2 besten Komponenten behält
pca_sklearn = PCA(n_components=2)
# Maschine trainieren (fit) UND Daten umwandeln (transform)
# Fit: Maschine lernt Kovarianz, Eigenwerte etc. und speichert sie in 'pca_sklearn'
# Transform: Maschine spuckt die neuen Daten aus (abalone_pca_sklearn)
abalone_pca_sklearn = pca_sklearn.fit_transform(abalone_standardized)
# Vergleich der Ergebnisse
print("Manuell erklärte Varianz (ersten 2):", explained_variance[:2])
# Ausgabe: [78.70, 8.63]
# -> Das sind unsere Berechnungen: PC1 erklärt ~78.7% der Infos, PC2 ~8.6%.
print("Sklearn erklärte Varianz:", pca_sklearn.explained_variance_ratio_ * 100)
# Ausgabe: [78.70, 8.63]
# -> Die ".explained_variance_ratio_" Eigenschaft gehört zur MASCHINE (pca_sklearn),
#    weil sie das beim Lernen "verstanden" und gespeichert hat.
#    Das Ergebnis ist identisch (Beweis: Wir haben richtig gerechnet!)

---
## Phase 7: ML Modell (Bonus)
Einfaches Modell zum Vorhersagen des Alters (Rings)

In [ ]:
#Plot the Age (Rings) to see the ditribution
print (abalone ["Rings"].value_counts())
sns.displot(abalone ["Rings"], kde=True)

In [ ]:
#import libaries
from sklearn.tree import DecisionTreeRegressor, export_text, plot_tree
from sklearn.metrics import mean_squared_error, r2_score, classification_report,accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier

In [ ]:
#Standarize the data
x=abalone.drop ("Rings", axis=1)
y = abalone["Rings"]


scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)


In [ ]:
# train test split - Daten aufteilen
# Wir teilen die Daten: 80% zum Lernen (Train), 20% zum Testen (Test)
# random_state=42 sorgt dafür, dass wir immer die gleiche Aufteilung bekommen (reproduzierbar)
X_train, X_test, y_train, y_test = train_test_split(x_scaled, y, test_size=0.2, random_state=42)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print('Training sample of scaled independent variables:\n',  X_train) # training sample (independent variables)
print('Training sample of target variable:\n',  y_train) # training sample (independent variables)
print('Test sample of scaled independent variables:\n',  X_test) # training sample (independent variables)
print('Test sample of target variable:\n',  y_test) # training sample (independent variables)

## Theorie: Warum wir Lineare Regression nutzen
Um zu testen, ob PCA Informationen fehlerfrei bewahrt, nutzen wir bewusst ein **einfaches lineares Modell** und keine komplexen Verfahren wie Decision Trees.
**Der Vergleich (Analogie):**
*   **Lineare Regression ist wie ein Lineal 📏**: Sie prüft, ob die Daten schön linear angeordnet ("sortiert") sind. Da PCA die Daten nur dreht, ist das der fairste Test.
*   **Decision Tree ist wie ein dickes Regelbuch 🌳**: Er lernt komplexe "Wenn-Dann"-Regeln. Er könnte selbst aus schlechten PCA-Daten noch etwas lernen (Overfitting) und würde uns vorgaukeln, die PCA wäre besser als sie ist.
**Unser Ziel**: Wir wollen sehen, wie viel Information (R²) wir verlieren, wenn wir von 8 Features auf 2 reduzieren.

In [ ]:


# 1. Baseline Model (Alle Features)
# ---------------------------------------------
model = LinearRegression()
model.fit(X_train, y_train)
print(r2_score(y_test, model.predict(X_test)))

In [ ]:
# pca model - Lineare Regression mit PCA-Features
X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(abalone_pca_sklearn, y, test_size=0.2, random_state=42)
print("PCA Train shape:", X_train_pca.shape) # Sollte (..., 2) sein
print ("PCA test shape:", X_test_pca.shape)

In [ ]:
# evaluate - MSE, R², MAE vergleichen
model = LinearRegression()
model.fit(X_train_pca, y_train_pca)
print(r2_score(y_test_pca, model.predict(X_test_pca)))

In [ ]:
# 1. Wir erzeugen ein Gitter (Meshgrid) über den ganzen Plot-Bereich
x_min, x_max = abalone_pca_sklearn[:, 0].min() - 1, abalone_pca_sklearn[:, 0].max() + 1
y_min, y_max = abalone_pca_sklearn[:, 1].min() - 1, abalone_pca_sklearn[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                     np.arange(y_min, y_max, 0.1))

# 2. Wir lassen das Modell für jeden Punkt auf diesem Gitter das Alter schätzen
# ravel() macht aus dem Gitter eine lange Liste für das Modell
Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape) # Zurück in Gitter-Form für den Plot

# 3. Plotten
plt.figure(figsize=(10, 8))

# Der Hintergrund (die Modell-Vorhersage)
# contourf malt Flächen aus: Dunkel = Modell sagt "Jung", Hell = Modell sagt "Alt"
plt.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')

# Die echten Datenpunkte darüber
plt.scatter(abalone_pca_sklearn[:, 0], abalone_pca_sklearn[:, 1], c=y, cmap='viridis', edgecolor='k', s=20)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Lineare Regression: Vorhersage-Landschaft (Hintergrund) vs. Echte Daten (Punkte)')
plt.colorbar(label='Alter (Rings)')
plt.show()

## Strategiewechsel: Von Regression zu Klassifikation
Bisher haben wir versucht, das **exakte Alter** (z.B. "9.5 Jahre") vorherzusagen. Das ist extrem schwierig und führt oft zu ungenauen Ergebnissen ($R^2 \approx 0.5$).
**Die Lösung: Binning (Gruppenbildung)**
Statt der exakten Zahl interessiert uns in der Praxis oft nur: *"Ist die Muschel jung, mittelalt oder alt?"*
*   Wir teilen die Ringe in 3 Gruppen: **Jung (0-8)**, **Mittel (9-10)**, **Alt (11+)**.
*   Damit machen wir es dem Modell leichter, robuste Muster zu erkennen.



In [ ]:
# ---------------------------------------------------------
# 1. Start: Kategorien erstellen ("Binning")
# ---------------------------------------------------------
# Wir definieren die Grenzen (Wände) für unsere Schubladen:
# Schublade 1: 0 bis 8 Ringe
# Schublade 2: 8 bis 10 Ringe (Hier sind die meisten Tiere!)
# Schublade 3: 10 bis 30 Ringe
bins = [0, 8, 10, 30]
# Die Labels (Namen), die wir auf die 3 Schubladen kleben
group_names = ['Jung', 'Mittel', 'Alt']
# pd.cut erledigt die Arbeit:
# Es nimmt die Spalte 'y' (Ringe/Zahlen) und wirft jeden Wert in die passende Schublade.
# Aus der Zahl "9" wird also das Wort "Mittel".
y_bins = pd.cut(y, bins, labels=group_names)
# ---------------------------------------------------------
# 2. Split: Daten aufteilen
# ---------------------------------------------------------
# Wir teilen die Daten wieder in Training (Lernen) und Test (Prüfen).
# Input (X): Unsere PCA-Daten ("abalone_pca_sklearn") -> Das sind die Fragen.
# Output (y): Unsere NEUEN Kategorien ("y_bins") -> Das sind die Antworten.
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    abalone_pca_sklearn,  # Die PCA-Daten
    y_bins,               # Die neuen Gruppen (Jung/Mittel/Alt)
    test_size=0.2,        # 20% für den Test aufheben
    random_state=42       # Damit wir immer das gleiche Ergebnis kriegen
)
# ---------------------------------------------------------
# 3. Das Modell: Random Forest Classifier
# ---------------------------------------------------------
# Wir gründen einen "Rat der Experten" (Random Forest).
# n_estimators=200: Wir fragen 200 einzelne Entscheidungsbäume nach ihrer Meinung (Mehrheitswahl).
# max_depth=7:      Jeder Baum darf maximal 7 Fragen tief bohren (verhindert, dass er Daten auswendig lernt).
rfc_grouped = RandomForestClassifier(n_estimators=200, random_state=42, max_depth=7)
# Training: Die 200 Bäume schauen sich die Trainingsdaten an und lernen Regeln.
rfc_grouped.fit(X_train_c, y_train_c)
# Vorhersage: Jetzt müssen die Bäume für die Test-Daten (die sie noch nie gesehen haben) raten.
y_pred_c = rfc_grouped.predict(X_test_c)
# ---------------------------------------------------------
# 4. Zeugnisvergabe (Auswertung)
# ---------------------------------------------------------
# Wie oft lag der Rat richtig? (Prozentzahl)
print("Test Accuracy (Genauigkeit):", accuracy_score(y_test_c, y_pred_c))
# Detaillierter Bericht: Wie gut wurde "Jung", "Mittel" und "Alt" jeweils erkannt?
print(classification_report(y_test_c, y_pred_c))

## Deep Learning: Der manuelle Training-Loop
Hier bauen wir ein **Neuronales Netz** (MLPClassifier). Um zu verstehen, wie es lernt, nutzen wir einen manuellen Loop statt der automatischen `.fit()` Funktion.
**Was passiert im Code?**
1.  **Shuffle 🎲**: Am Anfang jeder "Epoche" (Runde) mischen wir die Daten, damit das Netz nicht die Reihenfolge auswendig lernt.
2.  **Mini-Batches 📦**: Wir füttern dem Netz nicht alle 3000 Tiere auf einmal, sondern kleine Häppchen (z.B. 32 Stück).
3.  **Partial Fit 🧠**: Das Netz lernt aus diesen 32 Beispielen (`partial_fit`), passt seine Gewichte an und vergisst dabei aber nicht, was es vorher gelernt hat.
4.  **Epochen**: Wir wiederholen das Ganze 50 Mal, bis der Fehler (Loss) klein genug ist.

In [ ]:
# 1. Prepare Data (Pandas -> Numpy & Text -> Numbers)
le = LabelEncoder()
y_train_num = le.fit_transform(y_train_c)  # Labels into 0, 1, 2
y_test_num = le.transform(y_test_c)
# Convert everything to Numpy arrays to allow slicing [0:32]
X_train_np = np.array(X_train_c)
y_train_np = np.array(y_train_num)
# 2. Setup
batch_size = 32
number_epochs = 50
all_classes = np.unique(y_train_num) # These are [0, 1, 2]
neural_net = MLPClassifier(
    hidden_layer_sizes=(64, 32), 
    activation='relu', 
    batch_size=batch_size,
    random_state=42
)
# 3. The Loop
train_loss_history = []
for epoch in range(1, number_epochs + 1):
    epoch_losses = []  # Collect loss for this epoch
    
    # Shuffle data at the beginning of each epoch (Important for good learning!)
    permutation = np.random.permutation(len(y_train_np))
    X_shuffled = X_train_np[permutation]
    y_shuffled = y_train_np[permutation]
    
    # Inner Loop: Mini-Batches
    # range(start, stop, step):
    # start = 0 (We start with the first Abalone)
    # stop = len(y_train_np) (We stop when the data ends)
    # step = batch_size (e.g. 32). We jump in steps of 32: 0, 32, 64...
    # "i" is therefore always the START of our current batch.
    for i in range(0, len(y_train_np), batch_size):
        
        # Slicing [i : i+batch_size]:
        # We cut a piece out of the list.
        # "Take everything from index i (Start) to index i+32 (End)".
        # Example Round 1: i=0. We take [0:32] -> Abalone 0 to 31.
        # Example Round 2: i=32. We take [32:64] -> Abalone 32 to 63.
        X_batch = X_shuffled[i : i+batch_size]
        y_batch = y_shuffled[i : i+batch_size]
        
        # Train (partial_fit)
        # IMPORTANT: 'classes' must contain ALL possible classes!
        neural_net.partial_fit(X_batch, y_batch, classes=all_classes)
        
        # Save loss (if available)
        # (At the beginning loss_ might not be available yet, so try/except or check)
        if hasattr(neural_net, 'loss_'):
            epoch_losses.append(neural_net.loss_)
    # Print average loss of the epoch
    if epoch_losses:
        avg_loss = np.mean(epoch_losses)
        train_loss_history.append(avg_loss)
        if epoch % 10 == 0: # Print only every 10 epochs (to keep it clear)
            print(f'Epoch {epoch:02d} | Loss: {avg_loss:.4f}')
# 4. Result
print("\nTraining finished!")
print("Test Accuracy:", accuracy_score(y_test_num, neural_net.predict(X_test_c)))

In [ ]:
# 1. Input X (Our PCA Data) (no drop because we did PCA and strictly speaking no Sex column exists)
X = abalone_pca_sklearn 
# 2. Target y (Take Sex column directly, as it is already numbers 0,1,2)
y_sex = abalone['Sex']
# 3. Split
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X, y_sex, test_size=0.2, random_state=42)
print("PCA Train shape:", X_train_s.shape)
print("Test shape:", y_train_s.shape)

When you write X_train_s[0:32], Pandas often thinks you mean you mean index names ("Find the row named '0'") and not the position ("Give me the first row").
If we did train_test_split before, the indices are jumbled (e.g., row 1 has index 452, row 2 has index 12).
A loop [0:32] would then crash because there are no indices 0 to 32 anymore.

In [ ]:
# 1. Convert data to Numpy (The Gender Set!)
X_train_np = np.array(X_train_s)
y_train_np = np.array(y_train_s)
# 2. Setup
batch_size = 32
number_epochs = 50
all_classes = np.unique(y_train_np) # [0, 1, 2]
neural_net = MLPClassifier(
    hidden_layer_sizes=(64, 32), 
    activation='relu', 
    batch_size=batch_size,
    random_state=42
)
# 3. The Loop
train_loss_history = []
for epoch in range(1, number_epochs + 1):
    epoch_losses = [] 
    
    # Shuffle (with the Gender data!)
    permutation = np.random.permutation(len(y_train_np))
    X_shuffled = X_train_np[permutation]
    y_shuffled = y_train_np[permutation]
    
    # Inner Loop
    for i in range(0, len(y_train_np), batch_size):
        X_batch = X_shuffled[i : i+batch_size]
        y_batch = y_shuffled[i : i+batch_size]
        
        neural_net.partial_fit(X_batch, y_batch, classes=all_classes)
        
        if hasattr(neural_net, 'loss_'):
            epoch_losses.append(neural_net.loss_)
            
    if epoch_losses:
        avg_loss = np.mean(epoch_losses)
        train_loss_history.append(avg_loss)
        if epoch % 10 == 0: 
            print(f'Epoch {epoch:02d} | Loss: {avg_loss:.4f}')
# 4. Result
print("\nTraining finished!")
# -- STEP A: Make Predictions --
# We let the model run twice:
# 1. The "Exam" (Test Data):
# These are data the model has NEVER seen before. 
# Shows if it really understood (Generalization).
y_pred_sex = neural_net.predict(X_test_s)
# 2. The "Homework Check" (Training Data):
# These are the data it has already practiced 50 times.
# Shows if it just memorized the solutions.
train_pred = neural_net.predict(X_train_s)
# -- STEP B: Compare Grades --
print("--- Overfitting Check ---")
# Here we compare: Is it much better at homework (training) than in the exam (test)?
# If yes -> Overfitting (Memorization).
# If no (similar values) -> All good!
print(f"Training Accuracy  {accuracy_score(y_train_s, train_pred):.2%}")
print(f"Test Accuracy :    {accuracy_score(y_test_s, y_pred_sex):.2%}")
print("-" * 30)
# -- STEP C: The Detailed Report --
# Breaks down how well Male, Female, and Infant were recognized.
print(classification_report(y_test_s, y_pred_sex, target_names=["M", "F", "I"]))

In [ ]:
# We pick ONE random Abalone from the test data
import random
# We do this 5 times (loop)
for i in range(5):
    # 1. We roll a row number (e.g., "Row 500")
    random_index = random.randint(0, len(X_test_s) - 1)
    # 2. ACCESS THE QUESTION (X)
    # X_test_s is our large Numpy table with the PCA values.
    # [random_index] grabs exactly THAT ONE row (e.g., [1.5, -0.3]).
    # .reshape(1, -1) packs it into an extra box [[...]] so Sklearn doesn't complain.
    sample_data = X_test_s[random_index].reshape(1, -1)
    # 3. ACCESS THE ANSWER (y)
    # y_test_s is our list with the solutions (0, 1, 2).
    # .iloc[random_index] ("Integer Location") grabs the element at position "random_index".
    true_gender_code = y_test_s.iloc[random_index]
    # 4. The model is asked
    # We feed the question (sample_data) into the net.
    # [0] gets the number from the result list (e.g., [0] -> 0).
    prediction_code = neural_net.predict(sample_data)[0]
    # 5. Translation & Output
    gender_map = {0: "Male", 1: "Female", 2: "Infant"}
    
    print(f"--- Abalone Row {random_index} ---")
    print(f"Model guesses: {gender_map[prediction_code]}")
    print(f"Real solution: {gender_map[true_gender_code]}")
    if prediction_code == true_gender_code:
        print("✅ Correct!")
    else:
        print("❌ Wrong!")
    print("-" * 20) # Separator

In [ ]:
# 1. Known Data (Training)
train_pred = neural_net.predict(X_train_s)
train_acc = accuracy_score(y_train_s, train_pred)
# 2. Unknown Data (Test)
test_pred = neural_net.predict(X_test_s)
test_acc = accuracy_score(y_test_s, test_pred)
print(f"Training Accuracy: {train_acc:.2%}")
print(f"Test Accuracy:     {test_acc:.2%}")
diff = train_acc - test_acc
if diff > 0.10: # More than 10% difference?
    print("⚠️ Warning: Suspected Overfitting!")
else:
    print("✅ All good. No strong overfitting.")